# 16 – Graph Nodes (Individual Unit Tests)

Tests every LangGraph node in isolation without running the full graph.  
Nodes: `pre_hook`, `supervisor_node`, `information_node`, `knowledge_node`,  
`metadata_node`, `capacity_node`, `rule_node`, `synthesizer_node`, `post_hook`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from graph.state import initial_state
from graph import nodes

## 1. pre_hook — guardrails + initialization

In [ ]:
state = initial_state(query='What is the retention GRR?')
result = nodes.pre_hook(state)

print('guardrail_passed:', result['guardrail_passed'])
print('query_id        :', result['query_id'])
print('start_time set  :', result['start_time'] > 0)
print('agent_results   :', result['agent_results'])  # reset to []
print('anomalies       :', result['anomalies'])       # reset to []

## 2. supervisor_node — intent + routing

In [ ]:
state = initial_state(query='What is the DQ score for retention?')
result = nodes.supervisor_node(state)

print('intent      :', result['intent'])
print('next_agents :', result['next_agents'])
print('data_products:', result.get('data_products'))

## 3. information_node

In [ ]:
state = initial_state(query='Show retention metrics', data_products=['retention'])
result = nodes.information_node(state)

ar = result['agent_results']
print(f'agent_results count : {len(ar)}')
if ar:
    r = ar[0]
    print(f'agent   : {r["agent"]}')
    print(f'success : {r["success"]}')
    metrics = (r.get('data') or {}).get('metrics', {})
    print(f'products: {list(metrics.keys())}')

## 4. knowledge_node

In [ ]:
state = initial_state(query='What are the governance policies?')
result = nodes.knowledge_node(state)

ar = result['agent_results']
if ar:
    r = ar[0]
    docs = (r.get('data') or {}).get('knowledge', [])
    print(f'agent={r["agent"]}  success={r["success"]}  docs={len(docs)}')
    for d in docs[:2]:
        print(f"  [{d['topic']}]")

## 5. metadata_node

In [ ]:
state = initial_state(query='Who owns the bookings dataset?', data_products=['bookings'])
result = nodes.metadata_node(state)

ar = result['agent_results']
if ar:
    r = ar[0]
    data = r.get('data') or {}
    print(f'agent={r["agent"]}  success={r["success"]}')
    if 'bookings' in data:
        print(f'  owner  : {data["bookings"].get("owner")}')
        print(f'  DQ     : {data["bookings"].get("data_quality", {}).get("score")}%')

## 6. capacity_node

In [ ]:
state = initial_state(query='Show open Jira tickets', data_products=['retention'])
result = nodes.capacity_node(state)

ar = result['agent_results']
if ar:
    r = ar[0]
    tickets = (r.get('data') or {}).get('tickets', [])
    print(f'agent={r["agent"]}  success={r["success"]}  tickets={len(tickets)}')

## 7. rule_node

In [ ]:
state = initial_state(query='list all rules')
result = nodes.rule_node(state)

ar = result['agent_results']
if ar:
    r = ar[0]
    rules = r.get('data') or []
    print(f'agent={r["agent"]}  success={r["success"]}  rules={len(rules)}')
    for rule in rules[:3]:
        print(f"  {rule['id']}: {rule['name']}")

## 8. synthesizer_node — LLM summary with fallback

In [ ]:
# Build a state with pre-filled agent_results
state = initial_state(
    query='Why did retention drop?',
    anomalies=['retention: GRR 78% below threshold 85%'],
)
state['agent_results'] = [
    {'agent': 'information', 'success': True, 'data': {'metrics': {'retention': {'grr': 78.0}}}},
    {'agent': 'metadata', 'success': True, 'data': {'retention': {'owner': 'cs-team@company.com'}}},
]

result = nodes.synthesizer_node(state)
print('Final summary:')
print(result['final_summary'])
print('\nConfidence:', result['confidence'])

## 9. post_hook — execution timing

In [ ]:
import time

state = initial_state(query='test')
state['start_time'] = time.perf_counter() - 0.5  # simulate 500ms earlier

result = nodes.post_hook(state)
print(f'execution_ms: {result["execution_ms"]}ms')
assert result['execution_ms'] >= 500, 'Should have recorded ~500ms'
print('Timing recorded correctly')